#Analisis de cambio de uso de suelo 2013-2017 completa

In [ ]:
# =============================================================================
# PALM Land Use/Land Cover Change Analysis
# =============================================================================
#
# Author: Juan Gabriel Loaiza
# Affiliation: Universidad Autónoma de Sinaloa (UAS), Mexico
# Project: Land-use/land-cover and water-quality analysis of the
#          Adolfo Lopez Mateos Reservoir (PALM), Sinaloa, Mexico
# Repository version: v1.0.0
#
# This notebook is part of the reproducibility materials associated with
# the corresponding scientific publication.
# =============================================================================

import rasterio
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
import os

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'data').is_dir() and (candidate / 'derived_data').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found. Run this notebook from within the PALM repository.')

REPO_ROOT = find_repo_root()
LULC_DIR = REPO_ROOT / 'derived_data' / 'lulc'
ZONAL_LULC_DIR = LULC_DIR / 'zones'
OUTPUT_DIR = REPO_ROOT / 'derived_data' / 'lulc_change'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')


In [ ]:
# ============================================================
# LAND USE AND LAND COVER CHANGE ANALYSIS
# Transition matrix: 2013–2015
# ============================================================

import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from IPython.display import display
from matplotlib.colors import ListedColormap, BoundaryNorm


# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(LULC_DIR / 'LULC_2013.tif')

raster_path_2 = str(LULC_DIR / 'LULC_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. IDENTIFY LAND USE AND LAND COVER CLASSES
# ============================================================

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=[
        f'Class {int(class_value)}'
        for class_value in classes
    ],
    columns=[
        f'Class {int(class_value)}'
        for class_value in classes
    ]
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=[
        f'Class {int(class_value)}'
        for class_value in classes
    ],
    columns=[
        f'Class {int(class_value)}'
        for class_value in classes
    ]
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = (
    pd.DataFrame(change_results)
    .set_index('Class')
)


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. LAND USE AND LAND COVER MAP SETTINGS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{raster_name_2}'
)

In [ ]:
# ============================================================
# LAND USE AND LAND COVER CHANGE ANALYSIS
# Transition matrix: 2015–2017
# ============================================================

import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from IPython.display import display
from matplotlib.colors import ListedColormap, BoundaryNorm


# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(LULC_DIR / 'LULC_2015.tif')

raster_path_2 = str(LULC_DIR / 'LULC_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. IDENTIFY LAND USE AND LAND COVER CLASSES
# ============================================================

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=[
        f'Class {int(class_value)}'
        for class_value in classes
    ],
    columns=[
        f'Class {int(class_value)}'
        for class_value in classes
    ]
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=[
        f'Class {int(class_value)}'
        for class_value in classes
    ],
    columns=[
        f'Class {int(class_value)}'
        for class_value in classes
    ]
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = (
    pd.DataFrame(change_results)
    .set_index('Class')
)


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. LAND USE AND LAND COVER MAP SETTINGS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 1 2013-2015

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona1_2013.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona1_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'
zone = 'zone_1'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analsis de cambio de uso de suelo ZONA 1 2015-2017

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona1_2015.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona1_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'
zone = 'zone_1'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 2 2013-2015

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona2_2013.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona2_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'
zone = 'zone_2'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 2 2015-2017

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona2_2015.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona2_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'
zone = 'zone_2'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 3 2013-2015

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona3_2013.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona3_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'
zone = 'zone_3'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 3 2015-2017

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona3_2015.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona3_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'
zone = 'zone_3'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 4 2013-2015

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona4_2013.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona4_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'
zone = 'zone_4'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 4 2015-2017

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona4_2015.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona4_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'
zone = 'zone_4'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 5 2013-2015

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona5_2013.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona5_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'
zone = 'zone_5'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 5 2015-2017

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona5_2015.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona5_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'
zone = 'zone_5'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 6 2013-2015

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona6_2013.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona6_2015.tif')

raster_name_1 = '2013'
raster_name_2 = '2015'
zone = 'zone_6'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)

#Analisis de cambio de uso de suelo ZONA 6 2015-2017

In [ ]:
# @title Land Use and Land Cover Change Analysis by Zone

# ============================================================
# 1. INPUT AND OUTPUT SETTINGS
# ============================================================

raster_path_1 = str(ZONAL_LULC_DIR / 'reclas_zona6_2015.tif')

raster_path_2 = str(ZONAL_LULC_DIR / 'reclas_zona6_2017.tif')

raster_name_1 = '2015'
raster_name_2 = '2017'
zone = 'zone_6'

period_years = 2

output_folder = str(OUTPUT_DIR)

os.makedirs(output_folder, exist_ok=True)

excel_output = os.path.join(
    output_folder,
    f'land_use_change_analysis_'
    f'{raster_name_1}_{raster_name_2}_{zone}.xlsx'
)


# ============================================================
# 2. RASTER READING FUNCTION
# ============================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        shape = data.shape

    return data, nodata, transform, crs, shape


# ============================================================
# 3. READ RASTERS
# ============================================================

raster_1, nodata_1, transform_1, crs_1, shape_1 = (
    read_raster(raster_path_1)
)

raster_2, nodata_2, transform_2, crs_2, shape_2 = (
    read_raster(raster_path_2)
)


# ============================================================
# 4. SPATIAL COMPATIBILITY CHECK
# ============================================================

if shape_1 != shape_2:
    raise ValueError(
        "The rasters do not have the same dimensions."
    )

if transform_1 != transform_2:
    print(
        "Warning: the rasters have different spatial "
        "alignment or resolution."
    )

if crs_1 != crs_2:
    print(
        "Warning: the rasters have different coordinate "
        "reference systems."
    )


# ============================================================
# 5. VALID PIXEL SELECTION
# ============================================================

flat_1 = raster_1.flatten()
flat_2 = raster_2.flatten()

valid_mask = (flat_1 > 0) & (flat_2 > 0)

if nodata_1 is not None:
    valid_mask = valid_mask & (flat_1 != nodata_1)

if nodata_2 is not None:
    valid_mask = valid_mask & (flat_2 != nodata_2)

flat_1 = flat_1[valid_mask]
flat_2 = flat_2[valid_mask]


# ============================================================
# 6. LAND USE AND LAND COVER CLASSES
# ============================================================

class_names = {
    1: 'Water',
    2: 'Forest',
    3: 'Agricultural land',
    4: 'Tropical forests',
    5: 'Bare land'
}

classes = np.unique(
    np.concatenate((flat_1, flat_2))
)

classes.sort()

number_of_classes = len(classes)

class_dictionary = {
    class_value: position
    for position, class_value in enumerate(classes)
}

matrix_labels = [
    f'Class {int(class_value)} - '
    f'{class_names.get(int(class_value), "Unknown")}'
    for class_value in classes
]


# ============================================================
# 7. TRANSITION MATRIX IN PIXELS
# ============================================================

transition_matrix = np.zeros(
    (number_of_classes, number_of_classes),
    dtype=int
)

for initial_class, final_class in zip(flat_1, flat_2):

    row = class_dictionary[initial_class]
    column = class_dictionary[final_class]

    transition_matrix[row, column] += 1


transition_matrix_pixels = pd.DataFrame(
    transition_matrix,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_pixels.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_pixels.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 8. TRANSITION MATRIX IN HECTARES
# ============================================================

pixel_area_ha = 30 * 30 / 10000

transition_matrix_ha = (
    transition_matrix * pixel_area_ha
)

transition_matrix_hectares = pd.DataFrame(
    transition_matrix_ha,
    index=matrix_labels,
    columns=matrix_labels
)

transition_matrix_hectares.index.name = (
    f'Initial class ({raster_name_1})'
)

transition_matrix_hectares.columns.name = (
    f'Final class ({raster_name_2})'
)


# ============================================================
# 9. LAND USE AND LAND COVER CHANGE INDICATORS
# ============================================================

change_results = []

for i, class_value in enumerate(classes):

    row = transition_matrix[i, :]
    column = transition_matrix[:, i]

    initial_area = row.sum()
    final_area = column.sum()

    persistence = transition_matrix[i, i]

    loss = initial_area - persistence
    gain = final_area - persistence

    net_change = gain - loss
    total_change = gain + loss

    change_percentage = (
        total_change / initial_area * 100
        if initial_area > 0
        else 0
    )

    annual_rate = (
        change_percentage / period_years
        if period_years > 0
        else 0
    )

    change_results.append({
        'Class': int(class_value),
        'Class Name': class_names.get(
            int(class_value),
            'Unknown'
        ),
        'Initial Area (ha)': (
            initial_area * pixel_area_ha
        ),
        'Final Area (ha)': (
            final_area * pixel_area_ha
        ),
        'Persistence (ha)': (
            persistence * pixel_area_ha
        ),
        'Loss (ha)': (
            loss * pixel_area_ha
        ),
        'Gain (ha)': (
            gain * pixel_area_ha
        ),
        'Net Change (ha)': (
            net_change * pixel_area_ha
        ),
        'Total Change (ha)': (
            total_change * pixel_area_ha
        ),
        'Change (%)': change_percentage,
        'Annual Rate (%)': annual_rate
    })


change_summary = pd.DataFrame(
    change_results
).set_index('Class')


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (pixels):"
)

display(transition_matrix_pixels)

print(
    f"\nTRANSITION MATRIX FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2} (ha):"
)

display(transition_matrix_hectares.round(2))

print(
    f"\nLAND USE AND LAND COVER CHANGE SUMMARY "
    f"FOR {zone.upper()} "
    f"{raster_name_1}–{raster_name_2}:"
)

display(change_summary.round(2))


# ============================================================
# 11. EXPORT RESULTS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_output,
    engine='openpyxl'
) as writer:

    transition_matrix_pixels.to_excel(
        writer,
        sheet_name='Transition_matrix_pixels'
    )

    transition_matrix_hectares.round(4).to_excel(
        writer,
        sheet_name='Transition_matrix_ha'
    )

    change_summary.round(4).to_excel(
        writer,
        sheet_name='Change_summary'
    )


print(
    "\nExcel file successfully exported to:"
)

print(excel_output)


# ============================================================
# 12. MAP COLORS AND CLASS LABELS
# ============================================================

colors = [
    'white',
    'blue',
    'darkgreen',
    'orange',
    'lightgreen',
    'gray'
]

labels = [
    '1. Water',
    '2. Forest',
    '3. Agricultural land',
    '4. Tropical forests',
    '5. Bare land'
]

color_map = ListedColormap(colors)

boundaries = np.arange(0, 7) - 0.5

normalization = BoundaryNorm(
    boundaries,
    color_map.N
)


# ============================================================
# 13. FUNCTION TO DISPLAY CLASSIFIED MAPS
# ============================================================

def plot_colored_classification(data, title):

    plt.figure(figsize=(8, 6))

    image = plt.imshow(
        data,
        cmap=color_map,
        norm=normalization
    )

    color_bar = plt.colorbar(
        image,
        ticks=np.arange(1, 6)
    )

    color_bar.set_label(
        'Land use and land cover class'
    )

    color_bar.ax.set_yticklabels(labels)

    plt.title(title)
    plt.axis('off')
    plt.show()


# ============================================================
# 14. DISPLAY RECLASSIFIED MAPS
# ============================================================

plot_colored_classification(
    raster_1,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_1}'
)

plot_colored_classification(
    raster_2,
    f'Reclassified land use and land cover map — '
    f'{zone.replace("_", " ").title()}, '
    f'{raster_name_2}'
)